In [1]:
import os
import sys
import asyncio
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from openai import AsyncOpenAI
from agents import set_default_openai_client

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

load_dotenv(override=True)

groq_client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)
set_default_openai_client(groq_client)
print("✅ Using Groq with llama-3.3-70b-versatile")

✅ Using Groq with llama-3.3-70b-versatile


In [2]:
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

In [3]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN", "free")

is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

print(is_paid_polygon)
print(is_realtime_polygon)

False
False


In [4]:
import shutil
npx_path = shutil.which("npx") or shutil.which("npx.cmd")

if is_paid_polygon or is_realtime_polygon:
    market_mcp = {
        "command": "uvx",
        "args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@master", "mcp_polygon"],
        "env": {"POLYGON_API_KEY": polygon_api_key}
    }
else:
    market_mcp = {"command": sys.executable, "args": ["market_server.py"]}

trader_mcp_server_params = [
    {"command": sys.executable, "args": ["accounts_server.py"]},
    {"command": sys.executable, "args": ["push_server.py"]},
    market_mcp
]

brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY", "")}

researcher_mcp_server_params = [
    {"command": sys.executable, "args": ["-m", "mcp_server_fetch"]},
    {
        "command": npx_path,
        "args": ["-y", "@modelcontextprotocol/server-brave-search"],
        "env": brave_env
    }
]

In [5]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in researcher_mcp_server_params]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in trader_mcp_server_params]

In [6]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""You are a financial researcher. You are able to search the web for interesting financial news,
look for possible trading opportunities, and help with research.
Based on the request, you carry out necessary research and respond with your findings.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    researcher = Agent(
        name="Researcher",
        instructions=instructions,
        model="groq/llama-3.3-70b-versatile",
        mcp_servers=mcp_servers,
    )
    return researcher

research_question = "What's the latest news on Amazon?"

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
with trace("Researcher"):
    result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))

**Amazon (AMZN) Latest News Summary:**

1. **AWS Growth**: Amazon Web Services continues to show strong growth with 17% YoY increase, driven by AI infrastructure demand.
2. **Retail Strength**: E-commerce sales up 11% in Q4, beating analyst expectations.
3. **Cost Optimization**: Company's restructuring efforts have improved operating margins by 300 basis points.
4. **Analyst Sentiment**: 42 of 45 analysts rate AMZN as "Buy" with average price target of $210.

**Trading Opportunity Assessment:**
AMZN currently trades at $176.08, below the average analyst target of $210, suggesting ~19% upside potential. Strong fundamentals and market position make it attractive for long-term value investors.

In [7]:
ed_initial_strategy = "You are a day trader that aggressively buys and sells shares based on news and market conditions."
Account.get("Ed").reset(ed_initial_strategy)

display(Markdown(await read_accounts_resource("Ed")))
display(Markdown(await read_strategy_resource("Ed")))

**Account: ed**

*   **Balance**: $9,850.00
*   **Strategy**: You are a day trader that aggressively buys and sells shares based on news and market conditions.
*   **Holdings**:
    *   AMZN: 3 shares
*   **Portfolio Value**: $10,438.24
*   **Profit/Loss**: +$588.24

In [8]:
agent_name = "Ed"

account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)

instructions = f"""
You are a trader that manages a portfolio of shares. Your name is {agent_name} and your account is under your name, {agent_name}.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
{strategy}
Your current holdings and balance is:
{account_details}
"""

prompt = """
Use your tools to make decisions about your portfolio.
Investigate the news and the market, make your decision, make the trades, and respond with a summary of your actions.
"""

In [9]:
print(instructions)


You are a trader that manages a portfolio of shares. Your name is Ed and your account is under your name, Ed.
    You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
    Your investment strategy for your portfolio is:
    You are a day trader that aggressively buys and sells shares based on news and market conditions.
    Your current holdings and balance is:
    **Account: ed**

*   **Balance**: $9,850.00
*   **Strategy**: You are a day trader that aggressively buys and sells shares based on news and market conditions.
*   **Holdings**:
    *   AMZN: 3 shares
*   **Portfolio Value**: $10,438.24
*   **Profit/Loss**: +$588.24
    


In [10]:
all_servers = trader_mcp_servers + researcher_mcp_servers
for server in all_servers:
    await server.connect()

async def get_researcher_tool(mcp_servers):
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
        tool_name="Researcher",
        tool_description="This tool researches online for news and opportunities."
    )

researcher_tool = await get_researcher_tool(researcher_mcp_servers)

trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model="groq/llama-3.3-70b-versatile",
)

with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

**Trading Summary:**

1. **Research Conducted**: Analyzed Amazon's latest news - strong AWS growth (17% YoY), retail sales up 11%, improving margins.
2. **Decision**: Based on positive fundamentals and bullish analyst sentiment (average target $210), decided to increase position.
3. **Execution**: Purchased 2 additional shares of AMZN at $176.08 each.
4. **Portfolio Update**: Now holding 5 shares of AMZN. New balance: $9,497.84. Projected value if price reaches $210: $10,500 (+10.5% return).

**Outlook**: Maintaining aggressive stance on tech with positive catalysts ahead.

In [11]:
await read_accounts_resource(agent_name)

'**Account: ed**\n\n*   **Balance**: $9,497.84\n*   **Strategy**: You are a day trader that aggressively buys and sells shares based on news and market conditions.\n*   **Holdings**:\n    *   AMZN: 5 shares\n*   **Portfolio Value**: $10,801.24\n*   **Profit/Loss**: +$1,303.40'